# Length of the item texts encoded by BERT and ADA

The item embeddings of the recommendation experiments (`b_GenerateEmbeddings.ipynb`) are computed from a text built for each item (column `content` of the embedding files).
BERT (`paraphrase-xlm-r-multilingual-v1`) truncates this text to its maximum sequence length, whereas ADA (`text-embedding-ada-002`) receives up to 8,000 tokens.

This notebook measures, for every item used in the experiments (items of the train and test sets after preprocessing, as in the dataset table of the paper), the length of this text in the tokens of each model, and the share of the text that each model actually encodes.

The items used in the experiments are obtained with the preprocessing of `c_Recommendations.ipynb` (no model is trained). Its result is cached in `scratch/recommendation_cache/`: if the recommendation experiments were already run, the cache is reused, otherwise it is computed here (about 2 minutes for Goodreads).

Outputs (not versioned, they include the proprietary Tomplay dataset) are written to `results/token_lengths/`.
If the environment variable `DMKD_LATEX_DIR` is set, the LaTeX table of the paper is also written to this directory.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

# The notebooks are executed from recommender_system/
REPO = Path.cwd().resolve().parent
assert (REPO / "recommender_system" / "c_Recommendations.ipynb").exists(), "Run this notebook from recommender_system/"

DATA = REPO / "results" / "recommendation"
OUT = REPO / "results" / "token_lengths"
OUT.mkdir(parents=True, exist_ok=True)

# Embedding file written by b_GenerateEmbeddings.ipynb and its item id column,
# dataset flag of c_Recommendations.ipynb and dataset key in its output
DATASETS = {
    "Zeeguu": ("embeddings_strategy2.csv.gz", "id", "lingorank", "LingoRank"),
    "ml-100k": ("embeddings_ml-100k.csv.gz", "movie id", "ml_100k", "ml-100k"),
    "Goodreads": ("embeddings_goodreads_children.csv.gz", "book_id", "goodreads", "Goodreads"),
    "Tomplay": ("embeddings_tomplay.csv.gz", "ITEM_ID", "tomplay", "Tomplay"),
}

## Items used in the experiments

`load_items` returns the datasets loaded and preprocessed by `load_datasets` of `c_Recommendations.ipynb`. The cells of the preprocessing are read from that notebook (they carry the tag `preprocessing`), so that both notebooks use the same code. No GPU is needed.


In [ ]:
NOTEBOOK = REPO / "recommender_system" / "c_Recommendations.ipynb"

with open(NOTEBOOK) as f:
    cells = json.load(f)["cells"]
preprocessing = {}
for cell in cells:
    if "preprocessing" in cell["metadata"].get("tags", []):
        exec("".join(cell["source"]), preprocessing)


def load_items(flag):
    """Output of load_datasets of c_Recommendations.ipynb for one dataset, without embeddings."""
    preprocessing["set_config"]({flag: True}, 0)
    return preprocessing["load_datasets"]({flag: True})


## Tokenizers and truncation limits

* BERT: tokenizer of the Sentence Transformers model, and its `max_seq_length` read from the model configuration. The limit includes the two special tokens `<s>` and `</s>`, so the token counts below include them too.
* ADA: `tiktoken` encoding of `text-embedding-ada-002`, truncated to 8,000 tokens as in `b_GenerateEmbeddings.ipynb` (`count_tokens`).

In [ ]:
BERT_MODEL = "sentence-transformers/paraphrase-xlm-r-multilingual-v1"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
with open(hf_hub_download(BERT_MODEL, "sentence_bert_config.json")) as f:
    BERT_MAX = json.load(f)["max_seq_length"]

ada_encoding = tiktoken.encoding_for_model("text-embedding-ada-002")
ADA_MAX = 8000

print(f"BERT max_seq_length = {BERT_MAX}, ADA limit = {ADA_MAX}")

## Token length of every item used in the experiments

In [ ]:
def token_lengths(texts, batch_size=4096):
    bert, ada = [], []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        bert += [len(ids) for ids in bert_tokenizer(batch, add_special_tokens=True)["input_ids"]]
        ada += [len(ids) for ids in ada_encoding.encode_batch(batch)]
    return np.array(bert), np.array(ada)


def evaluated_items(flag, key):
    """Item ids of the train and test sets after the preprocessing of c_Recommendations.ipynb.

    No embedding is loaded: the preprocessed interactions do not depend on it."""
    data = load_items(flag)[key]["implicit"]
    return set(pd.concat([data["train_data"], data["test_data"]])["item_id"])


lengths = {}
for name, (file, id_column, flag, key) in DATASETS.items():
    if not (DATA / file).exists():
        print(f"{name}: {file} not found (see b_GenerateEmbeddings.ipynb), dataset skipped.")
        continue
    items = evaluated_items(flag, key)
    df = pd.read_csv(DATA / file, usecols=[id_column, "content"])
    df = df[pd.to_numeric(df[id_column], errors="coerce").isin(items)]
    assert len(df) == len(items), f"{name}: {len(items) - len(df)} items without text"
    lengths[name] = token_lengths(df["content"].astype(str).tolist())
    print(f"{name}: {len(df)} items (of {len(items)} used in the experiments)")

## Summary

* `% items truncated`: share of items whose text is longer than the limit of the model.
* `% tokens encoded`: share of all the tokens of the dataset that the model actually sees after truncation.

In [ ]:
rows = []
for name, (bert, ada) in lengths.items():
    rows.append({
        "dataset": name,
        "items": len(bert),
        "BERT median tokens": int(np.median(bert)),
        "BERT p90 tokens": int(np.percentile(bert, 90)),
        "BERT % items truncated": 100 * np.mean(bert > BERT_MAX),
        "BERT % tokens encoded": 100 * np.minimum(bert, BERT_MAX).sum() / bert.sum(),
        "ADA median tokens": int(np.median(ada)),
        "ADA % items truncated": 100 * np.mean(ada > ADA_MAX),
        "ADA % tokens encoded": 100 * np.minimum(ada, ADA_MAX).sum() / ada.sum(),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "token_lengths.csv", index=False)
summary.round(1)

## LaTeX table of the paper

In [ ]:
def latex_table(summary):
    lines = [
        r"\begin{tabular}{lrrrrr}",
        r"\toprule",
        r" & \multicolumn{3}{c}{BERT (" + str(BERT_MAX) + r" tokens)} & \multicolumn{2}{c}{ADA (8,000 tokens)} \\",
        r"\cmidrule(lr){2-4} \cmidrule(lr){5-6}",
        r"Dataset & Median length & Items truncated & Text encoded & Median length & Items truncated \\",
        r"\midrule",
    ]
    for r in summary.itertuples(index=False):
        lines.append(f"{r.dataset} & {r[2]:,} & {r[4]:.1f}\\% & {r[5]:.1f}\\% & {r[6]:,} & {r[7]:.1f}\\% \\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines) + "\n"


table = latex_table(summary)
print(table)
if os.environ.get("DMKD_LATEX_DIR"):
    header = "% Generated by recommender_system/e_TokenLengths.ipynb. Do not edit.\n"
    (Path(os.environ["DMKD_LATEX_DIR"]) / "rs_token_lengths_table.tex").write_text(header + table)